# Inferir
En esta lección aprenderás a inferir sentimientos y temas
a partir de reseñas de productos y artículos de noticias.

**¿Qué significa inferir?**
El modelo analiza un texto y deduce información que no está
explícitamente escrita — como detectar emociones, sentimientos,
temas o entidades mencionadas.

In [2]:
from helper import get_completion

## Texto de la reseña
Usaremos la reseña de una lámpara como ejemplo principal.
Esta reseña tiene aspectos positivos y negativos —
el modelo deberá inferir el sentimiento sin que nosotros
se lo digamos explícitamente.

In [1]:
resena_lampara = """
Necesitaba una lámpara bonita para mi dormitorio, y esta \
tenía almacenamiento adicional y un precio razonable. \
Llegó rápido. El cordón de la lámpara se rompió durante \
el transporte y la empresa con gusto me envió uno nuevo. \
Llegó en pocos días también. Fue fácil de armar. Tuve \
una pieza faltante, así que contacté a su soporte y muy \
rápidamente me enviaron la pieza faltante. Lumina me \
parece una gran empresa que se preocupa por sus clientes \
y productos.
"""

## 1. Sentimiento (positivo/negativo)

**Contexto:** El primer paso más básico — ¿el cliente está
satisfecho o no? Útil para filtrar rápidamente reseñas
negativas que necesitan atención inmediata.

In [3]:
prompt = f"""
¿Cuál es el sentimiento de la siguiente reseña de producto,
delimitada por triple comillas invertidas?

Texto de la reseña: ```{resena_lampara}```
"""

response = get_completion(prompt)
print(response)

El sentimiento de la reseña es positivo y satisfecho. El cliente destaca la funcionalidad, el precio razonable, la rapidez en la entrega y el buen servicio al cliente por parte de la empresa.


### Versión concisa — respuesta en una sola palabra

**Línea nueva agregada:**
> "Da tu respuesta en una sola palabra: positivo o negativo"

**¿Por qué?** Al procesar miles de reseñas en código,
una sola palabra es más fácil de contabilizar y procesar
que una oración completa.

In [4]:
prompt = f"""
¿Cuál es el sentimiento de la siguiente reseña de producto,
delimitada por triple comillas invertidas?

Da tu respuesta en una sola palabra: "positivo" o "negativo".

Texto de la reseña: ```{resena_lampara}```
"""

response = get_completion(prompt)
print(response)

positivo


## 2. Identificar emociones

**Contexto:** Más detallado que solo positivo/negativo —
identifica exactamente qué emociones expresa el cliente.
Útil para entender profundamente la experiencia del usuario.

In [5]:
prompt = f"""
Identifica una lista de emociones que el escritor de \
la siguiente reseña está expresando. No incluyas más de \
cinco elementos en la lista. Formatea tu respuesta como \
una lista de palabras en minúsculas separadas por comas.

Texto de la reseña: ```{resena_lampara}```
"""

response = get_completion(prompt)
print(response)

feliz, satisfecho, agradecido, impresionado, confiado


## 3. Identificar enojo

**Contexto:** En un ecommerce real, detectar clientes enojados
es crítico — permite al equipo de soporte priorizar esas
reseñas y contactar al cliente antes de que el problema escale.

Si alguien está muy enojado, merece atención inmediata.

In [6]:
prompt = f"""
¿El escritor de la siguiente reseña está expresando enojo? \
La reseña está delimitada por triple comillas invertidas. \
Da tu respuesta como "sí" o "no".

Texto de la reseña: ```{resena_lampara}```
"""

response = get_completion(prompt)
print(response)

No.


## 4. Extraer producto y marca

**Contexto:** En un ecommerce con miles de productos,
necesitas saber automáticamente de qué producto habla
cada reseña y quién lo fabricó — sin leerla manualmente.

La respuesta en JSON permite procesarla directamente en código.

In [7]:
prompt = f"""
Identifica los siguientes elementos del texto de la reseña:
- Producto comprado por el cliente
- Empresa que fabricó el producto

La reseña está delimitada por triple comillas invertidas. \
Formatea tu respuesta como un objeto JSON con \
"producto" y "marca" como claves.
Si la información no está presente, usa "desconocido" \
como valor.
Haz tu respuesta lo más corta posible.

Texto de la reseña: ```{resena_lampara}```
"""

response = get_completion(prompt)
print(response)

{
    "producto": "lámpara",
    "marca": "Lumina"
}


## 5. Hacer múltiples tareas en un solo prompt

**Contexto:** Hasta ahora usamos un prompt separado para cada
tarea — sentimiento, emociones, enojo, producto, marca.

Eso significa 4 llamadas a la API = 4 veces más costo y tiempo.

Con un solo prompt podemos extraer todo a la vez —
más eficiente y más barato.

In [8]:
prompt = f"""
Identifica los siguientes elementos del texto de la reseña:
- Sentimiento (positivo o negativo)
- ¿El cliente está expresando enojo? (verdadero o falso)
- Producto comprado por el cliente
- Empresa que fabricó el producto

La reseña está delimitada por triple comillas invertidas. \
Formatea tu respuesta como un objeto JSON con \
"sentimiento", "enojo", "producto" y "marca" como claves.
Si la información no está presente, usa "desconocido" \
como valor.
Haz tu respuesta lo más corta posible.
Formatea el valor de enojo como un booleano.

Texto de la reseña: ```{resena_lampara}```
"""

response = get_completion(prompt)
print(response)

{
    "sentimiento": "positivo",
    "enojo": false,
    "producto": "lámpara",
    "marca": "Lumina"
}


## 6. Inferir temas de un artículo

**Contexto:** Imagina que tienes un sitio de noticias con
miles de artículos. No puedes leerlos todos para clasificarlos
por tema. El modelo puede inferir automáticamente de qué
trata cada artículo.

In [9]:
articulo = """
En una encuesta reciente realizada por el gobierno,
se pidió a los empleados del sector público que calificaran
su nivel de satisfacción con el departamento en el que trabajan.
Los resultados revelaron que la NASA fue el departamento
más popular con una calificación de satisfacción del 95%.

Un empleado de la NASA, Juan García, comentó los hallazgos
diciendo: "No me sorprende que la NASA haya quedado en primer
lugar. Es un gran lugar para trabajar con personas increíbles
y oportunidades extraordinarias. Me enorgullece ser parte
de una organización tan innovadora."

Los resultados también fueron bien recibidos por el equipo
directivo de la NASA, con el director Carlos López diciendo:
"Estamos encantados de saber que nuestros empleados están
satisfechos con su trabajo en la NASA. Tenemos un equipo
talentoso y dedicado que trabaja incansablemente para
lograr nuestras metas."

La encuesta también reveló que la Administración del
Seguro Social tuvo la calificación de satisfacción más baja,
con solo el 45% de los empleados indicando que estaban
satisfechos con su trabajo. El gobierno se ha comprometido
a abordar las preocupaciones planteadas por los empleados.
"""

## Inferir 5 temas del artículo

**Línea clave:** Le pedimos al modelo que identifique
los temas en 1 o 2 palabras — así obtenemos etiquetas
cortas y útiles para clasificar artículos automáticamente.

In [10]:
prompt = f"""
Determina cinco temas que se están discutiendo en el \
siguiente texto, delimitado por triple comillas invertidas.

Haz cada tema de una o dos palabras.

Formatea tu respuesta como una lista de elementos \
separados por comas.

Texto: ```{articulo}```
"""

response = get_completion(prompt)
print(response)

1. Encuesta
2. Satisfacción
3. NASA
4. Administración
5. Compromiso


## 7. Alerta automática de temas — Zero-Shot Learning

**Contexto:** Imagina que tienes un sitio de noticias y
quieres notificar automáticamente cuando aparezca un artículo
sobre un tema específico — sin leerlo manualmente.

**Zero-Shot Learning:** El modelo clasifica sin haber sido
entrenado con ejemplos previos — solo con el prompt.

In [11]:
lista_temas = [
    "nasa", "gobierno local", "ingenieria",
    "satisfaccion laboral", "gobierno federal"
]

prompt = f"""
Determina si cada elemento de la siguiente lista de temas \
es un tema en el texto a continuación, delimitado por \
triple comillas invertidas.

Da tu respuesta de la siguiente forma:
elemento de la lista: 0 o 1

Lista de temas: {", ".join(lista_temas)}

Texto: ```{articulo}```
"""

response = get_completion(prompt)
print(response)

- nasa: 1
- gobierno local: 0
- ingenieria: 0
- satisfaccion laboral: 1
- gobierno federal: 1


In [13]:
topics_dict = {i.split(': ')[0].strip().replace('- ', ''): int(i.split(': ')[1]) for i in response.strip().split(sep='\n')}

if topics_dict['nasa'] == 1:
    print("ALERTA: ¡Nuevo artículo sobre la NASA!")

ALERTA: ¡Nuevo artículo sobre la NASA!
